In [1]:
from DataProcessor import DataProcessor
import matplotlib.pyplot as plt
import numpy as np

DEFAULT_DURATION = 4.0
dp = DataProcessor("rawdata/X22/")
files = ['rawdata/X22/normal_gehen3.pickle', 'rawdata/X22/schnell_Laufen10.pickle', 'rawdata/X22/rennen1.pickle'];
titles = ['normales Gehen', 'schnelles Gehen ', 'Rennen'];
fs = dp.fs
fig, axes = plt.subplots(6, 1, figsize=(10, 8), sharex=True)

def find_activity_start(acc_x, acc_y, acc_z, fs, threshold, window_seconds):
    """
    Findet den Index ab dem die Bewegung beginnt.
    Berechnet die Gesamtbeschleunigung (ohne Gravitation) und sucht
    den ersten Moment wo die Energie einen Schwellwert überschreitet.
    """
    window = int(window_seconds * fs)
    
    # Mittlere Beschleunigung (Gravitations-Offset) entfernen
    acc_x_centered = acc_x - np.mean(acc_x)
    acc_y_centered = acc_y - np.mean(acc_y)
    acc_z_centered = acc_z - np.mean(acc_z)
    
    # Betrag der Gesamtbeschleunigung
    magnitude = np.sqrt(acc_x_centered**2 + acc_y_centered**2 + acc_z_centered**2)
    
    # Gleitender Mittelwert (lokale Energie)
    energy = np.convolve(magnitude, np.ones(window)/window, mode='same')
    
    # Ersten Index suchen wo Energie > Schwellwert
    indices = np.where(energy > threshold)[0]
    
    if len(indices) == 0:
        return 0  # kein Schwellwert gefunden → von Anfang an
    
    return indices[0]

#Hilfsfunktion: eine Messung laden und Zeit + alle drei Achsen zurückgeben
def calc_time_and_axis_values():
    
    treshold = 0.4
    window_seconds = 0.1
    
     #Zeitvektor t (in Sekunden) aus dem Acceleration-DataFrame holen
    t_acc = dp.dfAcc["t"].values
    t_gyr = dp.dfGyr["t"].values
        
    #x-, y- und z-Komponente der Beschleunigung holen
    x_acc = dp.dfAcc["x"].values
    y_acc = dp.dfAcc["y"].values
    z_acc = dp.dfAcc["z"].values
    
    x_gyr = dp.dfGyr["x"].values
    y_gyr = dp.dfGyr["y"].values
    z_gyr = dp.dfGyr["z"].values
    
    
    #Analyse erstes sample, dass Bewegung beinhaltet
    start_idx = find_activity_start(x_acc, y_acc, z_acc, fs, treshold, window_seconds)
    
    #Anzahl samples berechnen
    num_samples_1 = int(DEFAULT_DURATION*fs)
    #num_samples_2= int(DEFAULT_DURATION*fs)
    
    #letzes sample berechnen
    end_idx = min(start_idx + num_samples_1, len(x_acc))
    
    #Werte auf 
    t_acc = t_acc[start_idx:end_idx]
    
    #Setzen des ersten Bewegungsverts auf index 0 der Plot-Achse
    t_acc = t_acc - t_acc[0]
    
    x_acc = x_acc[start_idx:end_idx]
    y_acc = y_acc[start_idx:end_idx]
    z_acc = z_acc[start_idx:end_idx]
    
    t_gyr = t_gyr[start_idx:end_idx]
    t_gyr = t_gyr - t_gyr[0]
    
    x_gyr = x_gyr[start_idx:end_idx]
    y_gyr = y_gyr[start_idx:end_idx]
    z_gyr = z_gyr[start_idx:end_idx]
    
    return t_acc, x_acc, y_acc, z_acc, t_gyr, x_gyr, y_gyr, z_gyr

for i in range(3): 
    fileName = files[i]
    dp.loadRawData(fileName)
    for dev in dp.getDevices():
        dp.loadRawDataDevice(dev)
            
    t1, acc_x, acc_y, acc_z, t2, gyr_x, gyr_y, gyr_z =  calc_time_and_axis_values()
        
    #Drei Unterplots untereinander erstellen
    axes[i*2].plot(t1, acc_x, label="x")
    axes[i*2].plot(t1, acc_y, label="y")
    axes[i*2].plot(t1, acc_z, label="z")
    axes[i*2].set_ylabel("Acceleration [g]")
    axes[i*2].set_title(titles[i])
    axes[i*2].legend()
    axes[i*2].grid(True)

    axes[i*2+1].plot(t2, gyr_x, label="x")
    axes[i*2+1].plot(t2, gyr_y, label="y")
    axes[i*2+1].plot(t2, gyr_z, label="z")
    axes[i*2+1].set_ylabel("Gyros. [deg/s]")
    axes[i*2+1].legend()
    axes[i*2+1].grid(True)
    
    axes[5].set_xlabel("Zeit [s]")
    plt.tight_layout()
plt.show()

c:\Users\elyes\anaconda3\envs\DSP_Projekt\Lib\site-packages\vpython\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


<IPython.core.display.Javascript object>

In [16]:
def fft_spectrum(x):
    N = len(x)
    X = np.fft.rfft(x)
    freqs = np.fft.rfftfreq(N, 1/fs)
    amplitude = np.abs(X)          # nur Betrag
    return freqs, amplitude

fig, axes = plt.subplots(6, 1, figsize=(10, 14), sharex=True)

for i in range(3):
    dp.loadRawData(files[i])
    only_device = list(dp.data.keys())[0]
    dp.loadRawDataDevice(only_device)

    t1, acc_x, acc_y, acc_z, t2, gyr_x, gyr_y, gyr_z = calc_time_and_axis_values()

    # FFT berechnen
    freqs, fft_x_acc = fft_spectrum(acc_x)
    freqs, fft_y_acc = fft_spectrum(acc_y)
    freqs, fft_z_acc = fft_spectrum(acc_z)

    freqs, fft_x_gyr = fft_spectrum(gyr_x)
    freqs, fft_y_gyr = fft_spectrum(gyr_y)
    freqs, fft_z_gyr = fft_spectrum(gyr_z)

    # Acc FFT-Plot
    axes[i*2].plot(freqs, fft_x_acc, label="x")
    axes[i*2].plot(freqs, fft_y_acc, label="y")
    axes[i*2].plot(freqs, fft_z_acc, label="z")
    axes[i*2].set_ylabel("|X(f)| [g]")
    axes[i*2].set_title(titles[i])
    axes[i*2].legend()
    axes[i*2].grid(True)

    # Gyr FFT-Plot
    axes[i*2+1].plot(freqs, fft_x_gyr, label="x")
    axes[i*2+1].plot(freqs, fft_y_gyr, label="y")
    axes[i*2+1].plot(freqs, fft_z_gyr, label="z")
    axes[i*2+1].set_ylabel("|X(f)| [deg/s]")
    axes[i*2+1].legend()
    axes[i*2+1].grid(True)

axes[5].set_xlabel("Frequenz [Hz]")
plt.tight_layout()
plt.show()

In [17]:
def periodogram(x):
    N = len(x)
    X = np.fft.rfft(x)                     # DFT des Signals (Da signal reel, rfft und nicht fft wie in Unterrichtsbeispielen)
    freqs = np.fft.rfftfreq(N, (1/fs))   # Frequenzachse in Hz: f_k = k * fs / N
    Pxx = (1.0 / N) * np.abs(X) ** 2       # Periodogramm: (1/N) |X[k]|^2
    return freqs, Pxx

#def welch_periodogram(x, fs):


fig, axes = plt.subplots(6, 1, figsize=(10, 8), sharex=True)
#Frequenz-Plots: drei Aktivitäten untereinander, alle drei Achsen pro Plot
for i in range(3): 
    fileName = files[i]
    dp.loadRawData(fileName)
    for dev in dp.getDevices():
        dp.loadRawDataDevice(dev)
    
    t1, acc_x, acc_y, acc_z, t2, gyr_x, gyr_y, gyr_z =  calc_time_and_axis_values()
    
    freqs, Pxx_x_acc =  periodogram(acc_x)
    freqs, Pxx_y_acc =  periodogram(acc_y)
    freqs, Pxx_z_acc =  periodogram(acc_z)
    
    freqs, Pxx_x_gyr =  periodogram(gyr_x)
    freqs, Pxx_y_gyr =  periodogram(gyr_y)
    freqs, Pxx_z_gyr =  periodogram(gyr_z)
        
    #Drei Unterplots untereinander erstellen
    axes[i*2].semilogy(freqs, Pxx_x_acc, label="x")
    axes[i*2].semilogy(freqs, Pxx_y_acc, label="y")
    axes[i*2].semilogy(freqs, Pxx_z_acc, label="z")
    axes[i*2].set_ylabel("P_xx(f)")
    axes[i*2].set_title(titles[i])
    axes[i*2].legend()
    axes[i*2].grid(True)

    axes[i*2+1].semilogy(freqs, Pxx_x_gyr, label="x")
    axes[i*2+1].semilogy(freqs, Pxx_y_gyr, label="y")
    axes[i*2+1].semilogy(freqs, Pxx_z_gyr, label="z")
    axes[i*2+1].set_ylabel("[(deg/s)²/Hz]") #"P_ωω(f) [(deg/s)²/Hz]"
    axes[i*2+1].legend()
    axes[i*2+1].grid(True)
    axes[5].set_xlabel("Frequenz [Hz]")
    plt.tight_layout()
plt.show()

In [18]:
def welch_periodogram(x, segment_seconds, overlap_time):
    N = len(x)
    L = int(segment_seconds * fs)
    step = int(L*(1-overlap_time))
    
    segments_pxx = []
    start = 0
    
    while start + L <= N: 
        segment = x[start : start+L]
        window = np.hamming(L)
        
        segment_window = segment * window
        _, Pxx_seg = periodogram(segment_window)
        segments_pxx.append(Pxx_seg)
        start+=step
    Pxx_welch = np.mean(segments_pxx, axis=0)
    freqs = np.fft.rfftfreq(L, (1/fs))           
    return freqs, Pxx_welch

segment_time = 1
overlap_time = 0.5

fig, axes = plt.subplots(6, 1, figsize=(10, 8), sharex=True)
#Frequenz-Plots: drei Aktivitäten untereinander, alle drei Achsen pro Plot
for i in range(3): 
    fileName = files[i]
    dp.loadRawData(fileName)
    for dev in dp.getDevices():
        dp.loadRawDataDevice(dev)
    
    t1, acc_x, acc_y, acc_z, t2, gyr_x, gyr_y, gyr_z =  calc_time_and_axis_values()
    
    freq, Pxx_welch_x_acc =  welch_periodogram(acc_x, segment_time, overlap_time)
    freq, Pxx_welch_y_acc =  welch_periodogram(acc_y, segment_time, overlap_time)
    freq, Pxx_welch_z_acc =  welch_periodogram(acc_z, segment_time, overlap_time)
    
    freq, Pxx_welch_x_gyr =  welch_periodogram(gyr_x, segment_time, overlap_time)
    freq, Pxx_welch_y_gyr =  welch_periodogram(gyr_y, segment_time, overlap_time)
    freq, Pxx_welch_z_gyr =  welch_periodogram(gyr_z, segment_time, overlap_time)
        
    #Drei Unterplots untereinander erstellen
    axes[i*2].semilogy(freq, Pxx_welch_x_acc, label="x")
    axes[i*2].semilogy(freq, Pxx_welch_y_acc, label="y")
    axes[i*2].semilogy(freq, Pxx_welch_z_acc, label="z")
    axes[i*2].set_ylabel("P_xx(f)")
    axes[i*2].set_title(titles[i])
    axes[i*2].legend()
    axes[i*2].grid(True)

    axes[i*2+1].semilogy(freq, Pxx_welch_x_gyr, label="x")
    axes[i*2+1].semilogy(freq, Pxx_welch_y_gyr, label="y")
    axes[i*2+1].semilogy(freq, Pxx_welch_z_gyr, label="z")
    axes[i*2+1].set_ylabel("[(deg/s)²/Hz]") #"P_ωω(f) [(deg/s)²/Hz]"
    axes[i*2+1].legend()
    axes[i*2+1].grid(True)
    axes[5].set_xlabel("Frequenz [Hz]")
    plt.tight_layout()
plt.show()